# CLARA — Fase 3: Fine-Tuning a Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Aquest notebook realitza el fine-tuning de Llama 3 8B amb **QLoRA** i la llibreria **Unsloth** sobre el dataset d'instruccions legals andorranàs.

**Requisits:**
- Google Colab amb GPU (T4 o millor)
- Fitxer `train_data.jsonl` pujat a Colab o Google Drive

**Temps estimat:** 3-6 hores (Colab Free T4)

## 0b. Muntar Google Drive (checkpoints i recuperació de sessió)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Carpeta base del projecte al Drive
DRIVE_DIR = Path('/content/drive/MyDrive/CLARA')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f'Drive muntat. Checkpoints a: {CHECKPOINT_DIR}')

# Comprovar si hi ha checkpoints previs per reprendre
existing = sorted(CHECKPOINT_DIR.glob('checkpoint-*'))
if existing:
    print(f'\n✅ Checkpoints trobats: {[c.name for c in existing]}')
    print(f'   L\'entrenament es reprendrà des de: {existing[-1].name}')
    RESUME_FROM = str(existing[-1])
else:
    print('\nℹ️ Cap checkpoint previ. Entrenament des de zero.')
    RESUME_FROM = None


## 0. Verificar GPU i instal·lar dependències

In [ ]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Cap"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

if not torch.cuda.is_available():
    raise RuntimeError('⚠️ Cal GPU! Canvia el runtime: Entorn d\'execució → Canvia el tipus de runtime → T4 GPU')

In [ ]:
%%capture
!pip install unsloth transformers datasets trl peft bitsandbytes accelerate

## 1. Crear el dataset d'instruccions (si no el tens ja)

In [ ]:
# Exemples manuals d'alta qualitat
# Pots ampliar-los amb la generació automàtica del fase3_dataset_builder.py

EXAMPLES = [
    {
        'instruction': 'Quina és la pena prevista per al delicte de robatori amb violència segons el Codi Penal d\'Andorra?',
        'input': '',
        'output': 'Segons l\'article 237 del Codi Penal del Principat d\'Andorra, el robatori amb violència o intimidació en les persones és castigat amb pena de presó de dos a sis anys. Si la violència ha causat lesions greus a la víctima, la pena s\'eleva a entre quatre i vuit anys. Les circumstàncies agreujants previstes a l\'article 240 poden incrementar aquestes penes en un terç.'
    },
    {
        'instruction': 'Quins requisits cal complir per constituir una societat anònima a Andorra?',
        'input': '',
        'output': 'D\'acord amb la Llei 20/2007, del 18 d\'octubre, de societats anònimes i de responsabilitat limitada, per constituir una societat anònima a Andorra cal: (1) un capital social mínim de 60.000 euros, íntegrament subscrit i desemborsat en un 25% en el moment de la constitució; (2) un mínim d\'un accionista, que pot ser persona física o jurídica; (3) atorgament d\'escriptura pública davant notari andorrà; i (4) inscripció al Registre de Societats del Govern d\'Andorra.'
    },
    {
        'instruction': 'Quins drets té un treballador andorrà en cas d\'acomiadament improcedent?',
        'input': '',
        'output': 'Segons la Llei 35/2008, del 18 de desembre, del Codi de Relacions Laborals, en cas d\'acomiadament improcedent el treballador té dret a: (1) una indemnització de 15 dies de salari per any treballat, amb un màxim de 12 mensualitats; (2) el preavís corresponent o la seva indemnització substitutiva; i (3) el pagament de les vacances no gaudides. L\'empresa pot optar entre readmetre el treballador o abonar la indemnització.'
    },
    {
        'instruction': 'Quina és la definició de delicte d\'estafa al Codi Penal andorrà?',
        'input': '',
        'output': 'L\'article 226 del Codi Penal d\'Andorra defineix l\'estafa com l\'acció de qui, amb ànim de lucre i usant engany prou per produir error en un altre, el indueix a realitzar un acte de disposició en perjudici propi o d\'un tercer. La pena prevista és de presó d\'un a tres anys i multa de dotze a vint-i-quatre mesos. Si l\'import defraudat supera els 50.000 euros, la pena s\'incrementa.'
    },
    {
        'instruction': 'Com es regula la custòdia dels fills en cas de divorci a Andorra?',
        'input': '',
        'output': 'El Codi de Família d\'Andorra estableix que en cas de divorci, la custòdia dels fills menors es determina sempre atenent l\'interès superior del menor. El jutge pot atorgar la custòdia exclusiva a un progenitor o la custòdia compartida a ambdós. En tot cas, ambdós progenitors conserven la potestat parental. El progenitor no custodi té dret a un règim de visites adequat, i ambdós han de contribuir als aliments del menor de forma proporcional als seus ingressos.'
    },
]

import json
with open('train_data.jsonl', 'w', encoding='utf-8') as f:
    for ex in EXAMPLES:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f'Dataset creat: {len(EXAMPLES)} exemples')
print('⚠️ Recomanat: afegir 100-200 exemples de qualitat per a millors resultats')

## 2. Carregar model base amb Unsloth (QLoRA 4-bit)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3-8B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Auto-detect (bf16 si és compatible)
    load_in_4bit=True,   # QLoRA: redueix VRAM a ~6 GB
)

print(f'Model carregat. Paràmetres: {model.num_parameters() / 1e9:.1f}B')

## 3. Configurar l'adapter LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                       # Rank: 16 és un bon equilibri qualitat/memòria
    target_modules=[            # Mòduls on s'aplica LoRA
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # Optimització de memòria d'Unsloth
    random_state=42,
)

# Paràmetres entrenables vs totals
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paràmetres entrenables: {trainable:,} ({100 * trainable / total:.2f}% del total)')

## 4. Preparar dataset en format ChatML (Llama 3)

In [ ]:
from datasets import load_dataset

EOS = tokenizer.eos_token

def format_example(row):
    """Formata cada exemple en el format ChatML de Llama 3."""
    text = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n'
        'Ets CLARA, un expert jurídic especialitzat en la legislació del Principat d\'Andorra. '
        'Respon sempre en català correcte. Cita sempre l\'article i la llei corresponents.'
        '<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n'
        f"{row['instruction']}"
        '<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n'
        f"{row['output']}"
        f'{EOS}'
    )
    return {'text': text}

dataset = load_dataset('json', data_files='train_data.jsonl', split='train')
dataset = dataset.map(format_example)

print(f'Exemples al dataset: {len(dataset)}')
print('\nExemple formatat:')
print(dataset[0]['text'][:300] + '...')

## 5. Entrenament

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # Simula batch_size=8
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        warmup_steps=10,
        output_dir=str(CHECKPOINT_DIR),  # Checkpoints directament al Drive
        optim='adamw_8bit',
        save_strategy='steps',
        save_steps=100,                  # Checkpoint cada 100 steps (~15-20 min)
        save_total_limit=3,              # Conservar només els 3 últims (estalvia espai)
        logging_steps=5,
        report_to='none',               # Canvia a 'wandb' si vols tracking
        resume_from_checkpoint=RESUME_FROM,  # Reprendre si hi ha checkpoint previ
    ),
)

# Estadístiques de memòria pre-entrenament
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f'GPU = {gpu_stats.name}. VRAM màx = {max_memory} GB')
print(f'VRAM reservada = {start_gpu_memory} GB')

In [ ]:
# Iniciar entrenament
trainer_stats = trainer.train()

# Estadístiques post-entrenament
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f'\nVRAM usada durant el training: {used_memory} GB')
print(f'Temps total: {trainer_stats.metrics["train_runtime"] / 60:.1f} minuts')

## 6. Prova ràpida del model entrenat

In [ ]:
FastLanguageModel.for_inference(model)

def generate(question: str, max_tokens: int = 300) -> str:
    inputs = tokenizer(
        f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        f'Ets CLARA, expert jurídic d\'Andorra. Respon en català.<|eot_id|>'
        f'<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n',
        return_tensors='pt'
    ).to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1, do_sample=True)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print(generate('Quina és la pena per robatori a Andorra?'))

## 7. Desar l'adapter LoRA i descarregar-lo

In [ ]:
# Desar només l'adapter (pocs MB!)
model.save_pretrained('clara_lora_adapter')
tokenizer.save_pretrained('clara_lora_adapter')
print('Adapter desat a clara_lora_adapter/')

# Comprimir per descarregar
import shutil
shutil.make_archive('clara_lora_adapter', 'zip', 'clara_lora_adapter')
print('Arxiu comprimit: clara_lora_adapter.zip')

# Descarregar (Google Colab)
from google.colab import files
files.download('clara_lora_adapter.zip')

## 8. Fusió del model (opcional: per a conversió a GGUF)

Si vols fusionar l'adapter amb el model base per convertir-lo a GGUF (per a Ollama):

In [ ]:
# Fusionar pesos i desar en float16
# Atenció: requereix ~16 GB de VRAM o RAM, pot fallar en T4
# Recomanat: fer-ho en local si tens prou RAM

model.save_pretrained_merged(
    'clara_merged_16bit',
    tokenizer,
    save_method='merged_16bit'
)
print('Model fusionat desat a clara_merged_16bit/')

# Conversió a GGUF Q4_K_M (des de terminal, en local):
# python llama.cpp/convert_hf_to_gguf.py clara_merged_16bit --outtype q4_k_m
# → genera: clara_merged_16bit-Q4_K_M.gguf

pip install unsloth transformers
python -c "
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Meta-Llama-3-8B-Instruct-bnb-4bit', load_in_4bit=True
)
model.load_adapter('clara_lora_adapter')
model = model.merge_and_unload()
model.save_pretrained_merged('clara_merged_16bit', tokenizer, save_method='merged_16bit')
"

In [2]:
# %pip install transformers peft accelerate torch

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
import torch

login()  # Introdueix el token quan ho demani o passa token="hf_..."

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
ADAPTER_DIR = "../clara_lora_adapter"
OUTPUT_DIR = "clara_merged_16bit"

print("Carregant model base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
)

print("Aplicant adapter LoRA...")
model = PeftModel.from_pretrained(model, ADAPTER_DIR)

print("Fusionant pesos...")
model = model.merge_and_unload()

print("Desant model fusionat...")
model.save_pretrained(OUTPUT_DIR, safe_serialization=False, save_original_format=False)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Fet! Model desat a {OUTPUT_DIR}/")

Carregant model base...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Aplicant adapter LoRA...
Fusionant pesos...
Desant model fusionat...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fet! Model desat a clara_merged_16bit/


In [ ]:
from unsloth import FastLanguageModel

# 1. Carregar model base
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)

# 2. Afegir estructura LoRA al model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
)

# 3. Carregar els pesos de l'adapter
from peft import PeftModel
model = PeftModel.from_pretrained(model, '/content/clara_lora_adapter')
print("Adapter carregat!")

# 4. Fusionar i desar
model.push_to_hub_merged(
    "MSS136/llama-clara",  # substitueix pel teu usuari HF
    tokenizer,
    save_method='merged_16bit',
    token=None  # O especifica token="hf_..." si no has iniciat sessió amb login()
)
print("Fet!")

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "path/to/your/bnb-model",
    torch_dtype="float16",
    device_map="cpu",
)

model.save_pretrained("path/to/output-fp16")

cd C:\Users\marcs\Documents\GitHub\CLARA\llama.cpp

# 1. Descarregar el model de HuggingFace
python -c "
from huggingface_hub import snapshot_download
snapshot_download('MSS136/llama-clara', local_dir='../llama-clara-hf', token='hf_xxxxxxxxxxxx')
"

python -c "
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id='meta-llama/Llama-3.1-8B-Instruct',
    filename='tokenizer.model',
    local_dir='../llama-clara-hf',
    token='hf_xxxxxxxxxxxx'
)
print('Fet!')
"

# 2. Convertir a GGUF F16
python convert_hf_to_gguf.py ../llama-clara-hf --outtype f16 --outfile ../llama-clara-hf/clara-f16.gguf

# Des de on has extret el ZIP (ajusta el path)
.\llama-quantize.exe ..\llama-clara-hf\clara-f16.gguf ..\llama-clara-hf\clara-Q4_K_M.gguf Q4_K_M

Despres del modelfile

Arrancar Ollama
ollama serve

En una altra terminal, crear el model
cd C:\Users\marcs\Documents\GitHub\CLARA
ollama create clara -f Modelfile

Verificar que s'ha creat
ollama list